In [0]:
from pyspark.sql import functions as F

In [0]:
silver_train = spark.table("final_project.silver.train")

## Define outlier rules

In this section we explicitly define what we consider an outlier or invalid record. Typical checks:

* Fare amount range (0 < fare_amount ≤ 500)
* Passenger count range (1 ≤ passenger_count ≤ 6)
* NYC bounding box for pickup and dropoff coordinates
* Optional: additional rules on distances or derived features

In [0]:
outlier_condition = (
    F.col("fare_amount").isNull() |
    F.col("passenger_count").isNull() |
    F.col("pickup_longitude").isNull() |
    F.col("pickup_latitude").isNull() |
    F.col("dropoff_longitude").isNull() |
    F.col("dropoff_latitude").isNull() |
    F.col("pickup_datetime").isNull() |
    
    (F.col("fare_amount") <= 0) |
    (F.col("fare_amount") > 500) |

    (F.col("passenger_count") < 1) |
    (F.col("passenger_count") > 6) |
    
    (F.col("pickup_longitude")  < -75) | (F.col("pickup_longitude")  > -72) |
    (F.col("dropoff_longitude") < -75) | (F.col("dropoff_longitude") > -72) |
    (F.col("pickup_latitude")   <  40) | (F.col("pickup_latitude")   >  42) |
    (F.col("dropoff_latitude")  <  40) | (F.col("dropoff_latitude")  >  42)
)

In [0]:
outlier_count = silver_train.filter(outlier_condition).count()
print(f"Outliers in final_project.silver.train: {outlier_count}")

## Assertion checks (fail the job if data is bad)




Here we convert the data quality rules into assertions. 

If any assertion fails, the notebook raises an error, which causes the Databricks Job run to fail. This turns the notebook into an automatic gatekeeper for data quality.


In [0]:
assert outlier_count == 0, (
    f"Data quality check failed: {outlier_count} outliers found "
    "in final_project.silver.train"
)

print("Data quality check passed: no outliers in final_project.silver.train.")
